# День 2 — Hidden states и эмбеддинги

Цель: научиться получать скрытые состояния модели и извлекать CLS-эмбеддинг текста.

In [1]:
import numpy as np
import torch

from transformers import AutoTokenizer, AutoModel
from sklearn.metrics.pairwise import cosine_similarity

C:\ProgramData\anaconda3\envs\transformers_overall\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
model_name = "distilbert-base-uncased"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModel.from_pretrained(model_name)

model.eval()

print("Model:", model_name)
print("Vocab size:", tokenizer.vocab_size)
print("Max length:", tokenizer.model_max_length)

Loading weights: 100%|██████████| 100/100 [00:00<00:00, 9336.24it/s]
[transformers] DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_layer_norm.bias   | UNEXPECTED |  | 
vocab_transform.weight  | UNEXPECTED |  | 
vocab_transform.bias    | UNEXPECTED |  | 
vocab_projector.bias    | UNEXPECTED |  | 
vocab_layer_norm.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Model: distilbert-base-uncased
Vocab size: 30522
Max length: 512


In [3]:
def tokenize_texts(texts, max_length=128):
    return tokenizer(
        texts,
        padding=True,
        truncation=True,
        max_length=max_length,
        return_tensors="pt"
    )

In [4]:
text = "This movie was absolutely amazing!"

tokens = tokenizer(text, return_tensors="pt")

with torch.no_grad():
    outputs = model(**tokens)

print(type(outputs))
print("Last hidden state shape:", outputs.last_hidden_state.shape)

<class 'transformers.modeling_outputs.BaseModelOutput'>
Last hidden state shape: torch.Size([1, 8, 768])


In [5]:
cls_embedding = outputs.last_hidden_state[:, 0, :]

print("CLS embedding shape:", cls_embedding.shape)
print("Первые 5 значений:")
print(cls_embedding[0][:5])

CLS embedding shape: torch.Size([1, 768])
Первые 5 значений:
tensor([ 0.0682, -0.0685,  0.1918,  0.0139, -0.0544])


In [6]:
def get_embeddings(texts, tokenizer, model, batch_size=32, max_length=128):
    all_embeddings = []

    for i in range(0, len(texts), batch_size):
        batch_texts = texts[i:i + batch_size]

        tokens = tokenizer(
            batch_texts,
            padding=True,
            truncation=True,
            max_length=max_length,
            return_tensors="pt"
        )

        with torch.no_grad():
            outputs = model(**tokens)

        cls_embeddings = outputs.last_hidden_state[:, 0, :]
        all_embeddings.append(cls_embeddings.cpu().numpy())

    return np.vstack(all_embeddings)

In [7]:
texts = [
    "This movie was absolutely amazing!",
    "Terrible movie, waste of time.",
    "Pretty good, I liked it.",
    "Boring and too long."
]

embeddings = get_embeddings(texts, tokenizer, model)

print("Embeddings shape:", embeddings.shape)
print("Ожидается: (4, 768) для DistilBERT")

Embeddings shape: (4, 768)
Ожидается: (4, 768) для DistilBERT


In [8]:
def similarity(text1, text2, tokenizer, model):
    emb = get_embeddings([text1, text2], tokenizer, model)
    sim = cosine_similarity(emb[0:1], emb[1:2])[0][0]
    return sim

sim1 = similarity("Great movie!", "Amazing film!", tokenizer, model)
sim2 = similarity("Great movie!", "Terrible film!", tokenizer, model)

print(f"Сходство похожих: {sim1:.3f}")
print(f"Сходство разных: {sim2:.3f}")

Сходство похожих: 0.995
Сходство разных: 0.984


In [9]:
pairs = [
    ("Great movie!", "Amazing film!"),
    ("Great movie!", "Terrible film!"),
    ("I loved this movie.", "I hated this movie."),
    ("The cat is sleeping.", "A dog is running."),
    ("This movie was boring.", "This film was too long and dull."),
]

for text1, text2 in pairs:
    sim = similarity(text1, text2, tokenizer, model)
    print(f"{sim:.3f} | {text1} <> {text2}")

0.995 | Great movie! <> Amazing film!
0.984 | Great movie! <> Terrible film!
0.987 | I loved this movie. <> I hated this movie.
0.974 | The cat is sleeping. <> A dog is running.
0.983 | This movie was boring. <> This film was too long and dull.


## Выводы

- AutoModel возвращает hidden states для каждого токена.
- last_hidden_state имеет форму [batch_size, sequence_length, hidden_size].
- Для DistilBERT hidden_size равен 768.
- CLS embedding можно использовать как векторное представление текста.
- Косинусное сходство показывает близость эмбеддингов, но без fine-tuning оно не всегда идеально отражает тональность.
- Косинусное сходство на CLS-эмбеддингах не всегда хорошо отражает тональность: противоположные отзывы могут быть близки, если они говорят об одной теме.